In [1]:
# =========================
# BLOCK 1. LOAD DATA + CREATE FEATURES
# =========================

from google.colab import files
uploaded = files.upload()

import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("cirrhosis.csv")

# Outcome: death = 1 if Status == D, otherwise 0
df["death"] = np.where(df["Status"] == "D", 1, 0)

# Convert age from days to years
df["Age_years"] = (df["Age"] / 365.25).round(1)

# Feature engineering dataframe
df_fe = df.copy()

# Clinically engineered features
df_fe["Albumin_Bilirubin_Ratio"] = df_fe["Albumin"] / (df_fe["Bilirubin"] + 1e-6)
df_fe["High_Bilirubin"] = np.where(df_fe["Bilirubin"] > 1.2, 1, 0)
df_fe["Low_Albumin"] = np.where(df_fe["Albumin"] < 3.5, 1, 0)
df_fe["High_Prothrombin"] = np.where(df_fe["Prothrombin"] > 12, 1, 0)
df_fe["Low_Platelets"] = np.where(df_fe["Platelets"] < 150, 1, 0)

# Clinical complication count
df_fe["Ascites_Y"] = np.where(df_fe["Ascites"] == "Y", 1, 0)
df_fe["Hepatomegaly_Y"] = np.where(df_fe["Hepatomegaly"] == "Y", 1, 0)
df_fe["Spiders_Y"] = np.where(df_fe["Spiders"] == "Y", 1, 0)
df_fe["Edema_abnormal"] = np.where(df_fe["Edema"].isin(["S", "Y"]), 1, 0)

df_fe["Complication_Count"] = (
    df_fe["Ascites_Y"]
    + df_fe["Hepatomegaly_Y"]
    + df_fe["Spiders_Y"]
    + df_fe["Edema_abnormal"]
)

# Feature groups
lab_features = [
    "Bilirubin",
    "Cholesterol",
    "Albumin",
    "Copper",
    "Alk_Phos",
    "SGOT",
    "Tryglicerides",
    "Platelets",
    "Prothrombin"
]

clinical_features = [
    "Ascites",
    "Hepatomegaly",
    "Spiders",
    "Edema"
]

demographic_features = [
    "Age_years",
    "Sex"
]

treatment_features = [
    "Drug"
]

severity_features = [
    "Stage"
]

engineered_features = [
    "Albumin_Bilirubin_Ratio",
    "High_Bilirubin",
    "Low_Albumin",
    "High_Prothrombin",
    "Low_Platelets",
    "Complication_Count"
]

features_lab_only = lab_features
features_clinical_only = clinical_features
features_lab_clinical = lab_features + clinical_features

features_full = (
    demographic_features
    + treatment_features
    + clinical_features
    + lab_features
    + severity_features
)

features_full_engineered = features_full + engineered_features

feature_sets = {
    "Clinical-only": features_clinical_only,
    "Lab-only": features_lab_only,
    "Lab + Clinical": features_lab_clinical,
    "Full": features_full,
    "Full + Engineered": features_full_engineered
}

print("Data loaded successfully.")
print("Shape:", df_fe.shape)
print("\nOutcome distribution:")
print(df_fe["death"].value_counts())
print("\nFeature sets:")
for k, v in feature_sets.items():
    print(k, ":", len(v), "raw features")

Saving cirrhosis.csv to cirrhosis.csv
Data loaded successfully.
Shape: (418, 32)

Outcome distribution:
death
0    257
1    161
Name: count, dtype: int64

Feature sets:
Clinical-only : 4 raw features
Lab-only : 9 raw features
Lab + Clinical : 13 raw features
Full : 17 raw features
Full + Engineered : 23 raw features


In [2]:
# =========================
# BLOCK 2. LEAKAGE-FREE REPEATED CROSS-VALIDATION ANALYSIS
# =========================

!pip install xgboost -q

import pandas as pd
import numpy as np

from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    make_scorer
)
from xgboost import XGBClassifier

# 5-fold CV repeated 10 times = 50 splits
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

# Models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="liblinear"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )
}

# Scoring metrics
scoring = {
    "AUROC": "roc_auc",
    "AUPRC": "average_precision",
    "Accuracy": "accuracy",
    "Sensitivity": "recall",
    "F1": "f1"
}

def build_pipeline(X, model):
    """
    Build a leakage-free preprocessing + modeling pipeline.
    Numeric imputation medians are estimated only from the training fold.
    Categorical imputation and one-hot encoding categories are learned only from the training fold.
    Standardization parameters are estimated only from the training fold.
    Then the fitted preprocessing is applied to the held-out fold.
    """
    categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
    numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols)
        ]
    )

    pipeline = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ])

    return pipeline

all_results = []

# Main repeated CV
for feature_set_name, feature_list in feature_sets.items():
    X_current = df_fe[feature_list].copy()
    y = df_fe["death"].copy()

    for model_name, model in models.items():
        pipe = build_pipeline(X_current, model)

        cv_result = cross_validate(
            pipe,
            X_current,
            y,
            cv=cv,
            scoring=scoring,
            return_train_score=False,
            n_jobs=-1
        )

        all_results.append({
            "Feature_Set": feature_set_name,
            "Model": model_name,
            "N_Raw_Features": len(feature_list),
            "CV_Design": "5-fold stratified CV repeated 10 times (50 splits)",
            "AUROC_mean": cv_result["test_AUROC"].mean(),
            "AUROC_sd": cv_result["test_AUROC"].std(),
            "AUPRC_mean": cv_result["test_AUPRC"].mean(),
            "AUPRC_sd": cv_result["test_AUPRC"].std(),
            "Accuracy_mean": cv_result["test_Accuracy"].mean(),
            "Accuracy_sd": cv_result["test_Accuracy"].std(),
            "Sensitivity_mean": cv_result["test_Sensitivity"].mean(),
            "Sensitivity_sd": cv_result["test_Sensitivity"].std(),
            "F1_mean": cv_result["test_F1"].mean(),
            "F1_sd": cv_result["test_F1"].std()
        })

Table2_model_performance_corrected = pd.DataFrame(all_results)

# Round for display
Table2_display = Table2_model_performance_corrected.copy()
float_cols = Table2_display.select_dtypes(include=["float64"]).columns
Table2_display[float_cols] = Table2_display[float_cols].round(3)

Table2_display = Table2_display.sort_values(
    by=["AUROC_mean", "AUPRC_mean"],
    ascending=False
)

display(Table2_display)

# Final selected model:
# Logistic Regression using Full + Engineered features.
# Rationale: comparable AUROC with ensemble methods, better interpretability.
X_final = df_fe[features_full_engineered].copy()
y_final = df_fe["death"].copy()

final_pipe = build_pipeline(
    X_final,
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="liblinear"
    )
)

# Pooled out-of-fold predictions for ROC/calibration/confusion matrix.
# NOTE: This is one 5-fold CV, used for pooled curves/figures.
# Table 2 reports repeated 5-fold CV mean across 50 splits.
single_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=1,
    random_state=42
)

y_pred_prob = cross_val_predict(
    final_pipe,
    X_final,
    y_final,
    cv=single_cv,
    method="predict_proba"
)[:, 1]

y_pred = (y_pred_prob >= 0.5).astype(int)

pooled_auroc = roc_auc_score(y_final, y_pred_prob)
pooled_auprc = average_precision_score(y_final, y_pred_prob)
pooled_accuracy = accuracy_score(y_final, y_pred)
pooled_sensitivity = recall_score(y_final, y_pred)
pooled_f1 = f1_score(y_final, y_pred)
pooled_brier = brier_score_loss(y_final, y_pred_prob)
pooled_cm = confusion_matrix(y_final, y_pred)

print("\nFinal model pooled out-of-fold results for figures:")
print("Pooled AUROC:", round(pooled_auroc, 3))
print("Pooled AUPRC:", round(pooled_auprc, 3))
print("Pooled Accuracy:", round(pooled_accuracy, 3))
print("Pooled Sensitivity:", round(pooled_sensitivity, 3))
print("Pooled F1:", round(pooled_f1, 3))
print("Pooled Brier score:", round(pooled_brier, 3))
print("Confusion matrix:")
print(pooled_cm)

,Feature_Set,Model,N_Raw_Features,CV_Design,AUROC_mean,AUROC_sd,AUPRC_mean,AUPRC_sd,Accuracy_mean,Accuracy_sd,Sensitivity_mean,Sensitivity_sd,F1_mean,F1_sd
9,Full,Logistic Regression,17,5-fold stratified CV repeated 10 times (50 spl...,0.840,0.032,0.769,0.047,0.765,0.039,0.709,0.084,0.698,0.054
10,Full,Random Forest,17,5-fold stratified CV repeated 10 times (50 spl...,0.837,0.037,0.783,0.053,0.778,0.031,0.630,0.068,0.685,0.048
12,Full + Engineered,Logistic Regression,23,5-fold stratified CV repeated 10 times (50 spl...,0.836,0.035,0.765,0.050,0.775,0.040,0.723,0.092,0.710,0.062
11,Full,XGBoost,17,5-fold stratified CV repeated 10 times (50 spl...,0.834,0.035,0.784,0.048,0.772,0.035,0.642,0.080,0.683,0.055
13,Full + Engineered,Random Forest,23,5-fold stratified CV repeated 10 times (50 spl...,0.833,0.036,0.776,0.051,0.784,0.031,0.648,0.064,0.697,0.047
14,Full + Engineered,XGBoost,23,5-fold stratified CV repeated 10 times (50 spl...,0.830,0.035,0.778,0.048,0.771,0.034,0.644,0.077,0.682,0.054
6,Lab + Clinical,Logistic Regression,13,5-fold stratified CV repeated 10 times (50 spl...,0.818,0.037,0.752,0.056,0.771,0.039,0.691,0.076,0.698,0.054
7,Lab + Clinical,Random Forest,13,5-fold stratified CV repeated 10 times (50 spl...,0.815,0.038,0.760,0.051,0.761,0.034,0.616,0.070,0.664,0.051
3,Lab-only,Logistic Regression,9,5-fold stratified CV repeated 10 times (50 spl...,0.812,0.040,0.745,0.059,0.765,0.044,0.682,0.081,0.690,0.062
4,Lab-only,Random Forest,9,5-fold stratified CV repeated 10 times (50 spl...,0.805,0.042,0.751,0.053,0.750,0.035,0.612,0.081,0.652,0.056



Final model pooled out-of-fold results for figures:
Pooled AUROC: 0.826
Pooled AUPRC: 0.745
Pooled Accuracy: 0.773
Pooled Sensitivity: 0.739
Pooled F1: 0.715
Pooled Brier score: 0.169
Confusion matrix:
[[204  53]
 [ 42 119]]


In [3]:
# =========================
# BLOCK 3. TABLE 1 + SUPPLEMENTARY TABLES + EXPORT FILES
# =========================

from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact
import pandas as pd
import numpy as np

# -------------------------
# Corrected Table 1
# One P-value per categorical variable
# -------------------------

numeric_vars = [
    "Age_years",
    "Bilirubin",
    "Albumin",
    "Prothrombin",
    "Platelets",
    "Alk_Phos",
    "SGOT",
    "Stage",
    "Complication_Count"
]

categorical_vars = [
    "Sex",
    "Ascites",
    "Hepatomegaly",
    "Spiders",
    "Edema"
]

def median_iqr(series):
    series = series.dropna()
    median = series.median()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    return f"{median:.2f} ({q1:.2f}-{q3:.2f})"

def format_p(p):
    if pd.isnull(p):
        return ""
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"

table1_rows = []

# Numeric variables: Mann-Whitney U test
for var in numeric_vars:
    group0 = df_fe[df_fe["death"] == 0][var].dropna()
    group1 = df_fe[df_fe["death"] == 1][var].dropna()

    try:
        p_value = mannwhitneyu(group0, group1, alternative="two-sided").pvalue
    except:
        p_value = np.nan

    table1_rows.append({
        "Variable": var,
        "Non-death / censored (death=0)": median_iqr(group0),
        "Death (death=1)": median_iqr(group1),
        "P_value": format_p(p_value)
    })

# Categorical variables: one p-value per variable
for var in categorical_vars:
    temp = df_fe[[var, "death"]].copy()
    temp[var] = temp[var].fillna("Unknown")

    contingency = pd.crosstab(temp[var], temp["death"])

    try:
        if contingency.shape == (2, 2):
            p_value = fisher_exact(contingency)[1]
            test_name = "Fisher exact test"
        else:
            p_value = chi2_contingency(contingency)[1]
            test_name = "Chi-square test"
    except:
        p_value = np.nan
        test_name = ""

    categories = sorted(temp[var].unique())

    first_row = True

    for cat in categories:
        n0 = ((temp[var] == cat) & (temp["death"] == 0)).sum()
        d0 = (temp["death"] == 0).sum()
        pct0 = n0 / d0 * 100

        n1 = ((temp[var] == cat) & (temp["death"] == 1)).sum()
        d1 = (temp["death"] == 1).sum()
        pct1 = n1 / d1 * 100

        table1_rows.append({
            "Variable": f"{var} = {cat}",
            "Non-death / censored (death=0)": f"{n0} ({pct0:.1f}%)",
            "Death (death=1)": f"{n1} ({pct1:.1f}%)",
            "P_value": format_p(p_value) if first_row else ""
        })

        first_row = False

Table1_baseline_characteristics_corrected = pd.DataFrame(table1_rows)

print("Corrected Table 1:")
display(Table1_baseline_characteristics_corrected)

# -------------------------
# Supplementary Table S1: Hyperparameter configurations
# -------------------------

Supplementary_Table_S1_hyperparameters = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Hyperparameters": "max_iter=1000; class_weight=balanced; solver=liblinear",
        "Preprocessing": "Median imputation for numeric variables; Unknown category imputation and one-hot encoding for categorical variables; standardization for numeric variables; all fitted within training fold only"
    },
    {
        "Model": "Random Forest",
        "Hyperparameters": "n_estimators=300; random_state=42; class_weight=balanced",
        "Preprocessing": "Median imputation for numeric variables; Unknown category imputation and one-hot encoding for categorical variables; all fitted within training fold only"
    },
    {
        "Model": "XGBoost",
        "Hyperparameters": "n_estimators=300; learning_rate=0.03; max_depth=3; subsample=0.8; colsample_bytree=0.8; eval_metric=logloss; random_state=42",
        "Preprocessing": "Median imputation for numeric variables; Unknown category imputation and one-hot encoding for categorical variables; all fitted within training fold only"
    }
])

# -------------------------
# Supplementary Table S2: Confusion matrices across feature sets for Logistic Regression
# -------------------------

confusion_rows = []

for feature_set_name, feature_list in feature_sets.items():
    X_current = df_fe[feature_list].copy()
    y = df_fe["death"].copy()

    pipe = build_pipeline(
        X_current,
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            solver="liblinear"
        )
    )

    y_prob = cross_val_predict(
        pipe,
        X_current,
        y,
        cv=single_cv,
        method="predict_proba"
    )[:, 1]

    y_hat = (y_prob >= 0.5).astype(int)
    cm = confusion_matrix(y, y_hat)

    tn, fp, fn, tp = cm.ravel()

    confusion_rows.append({
        "Feature_Set": feature_set_name,
        "Model": "Logistic Regression",
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Accuracy": accuracy_score(y, y_hat),
        "Sensitivity": recall_score(y, y_hat),
        "F1": f1_score(y, y_hat)
    })

Supplementary_Table_S2_confusion_matrices = pd.DataFrame(confusion_rows)

float_cols = Supplementary_Table_S2_confusion_matrices.select_dtypes(include=["float64"]).columns
Supplementary_Table_S2_confusion_matrices[float_cols] = Supplementary_Table_S2_confusion_matrices[float_cols].round(3)

# -------------------------
# Supplementary Table S3: Calibration/Brier score across feature sets for Logistic Regression
# -------------------------

calibration_rows = []

for feature_set_name, feature_list in feature_sets.items():
    X_current = df_fe[feature_list].copy()
    y = df_fe["death"].copy()

    pipe = build_pipeline(
        X_current,
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            solver="liblinear"
        )
    )

    y_prob = cross_val_predict(
        pipe,
        X_current,
        y,
        cv=single_cv,
        method="predict_proba"
    )[:, 1]

    calibration_rows.append({
        "Feature_Set": feature_set_name,
        "Model": "Logistic Regression",
        "Pooled_AUROC_for_curve": roc_auc_score(y, y_prob),
        "Pooled_AUPRC_for_curve": average_precision_score(y, y_prob),
        "Brier_score": brier_score_loss(y, y_prob)
    })

Supplementary_Table_S3_calibration = pd.DataFrame(calibration_rows)

float_cols = Supplementary_Table_S3_calibration.select_dtypes(include=["float64"]).columns
Supplementary_Table_S3_calibration[float_cols] = Supplementary_Table_S3_calibration[float_cols].round(3)

print("\nSupplementary Table S1: Hyperparameters")
display(Supplementary_Table_S1_hyperparameters)

print("\nSupplementary Table S2: Confusion matrices")
display(Supplementary_Table_S2_confusion_matrices)

print("\nSupplementary Table S3: Calibration")
display(Supplementary_Table_S3_calibration)

# -------------------------
# Export all corrected tables
# -------------------------

Table1_baseline_characteristics_corrected.to_csv(
    "Table1_baseline_characteristics_corrected.csv",
    index=False
)

Table2_model_performance_corrected.to_csv(
    "Table2_model_performance_corrected.csv",
    index=False
)

Table2_display.to_csv(
    "Table2_model_performance_corrected_rounded.csv",
    index=False
)

Supplementary_Table_S1_hyperparameters.to_csv(
    "Supplementary_Table_S1_hyperparameters.csv",
    index=False
)

Supplementary_Table_S2_confusion_matrices.to_csv(
    "Supplementary_Table_S2_confusion_matrices.csv",
    index=False
)

Supplementary_Table_S3_calibration.to_csv(
    "Supplementary_Table_S3_calibration.csv",
    index=False
)

print("\nSaved corrected files:")
print("Table1_baseline_characteristics_corrected.csv")
print("Table2_model_performance_corrected.csv")
print("Table2_model_performance_corrected_rounded.csv")
print("Supplementary_Table_S1_hyperparameters.csv")
print("Supplementary_Table_S2_confusion_matrices.csv")
print("Supplementary_Table_S3_calibration.csv")

Corrected Table 1:


,Variable,Non-death / censored (death=0),Death (death=1),P_value
0,Age_years,48.80 (40.80-56.40),53.50 (46.60-61.30),<0.001
1,Bilirubin,1.00 (0.70-1.80),3.20 (1.40-7.10),<0.001
2,Albumin,3.60 (3.35-3.82),3.40 (3.08-3.65),<0.001
3,Prothrombin,10.30 (9.90-10.80),11.00 (10.47-11.80),<0.001
4,Platelets,258.50 (209.00-318.00),224.00 (158.00-312.50),0.003
5,Alk_Phos,1132.00 (800.50-1648.50),1664.00 (1029.00-2468.00),<0.001
6,SGOT,98.00 (73.19-133.30),134.85 (99.33-176.70),<0.001
7,Stage,3.00 (2.00-3.00),4.00 (3.00-4.00),<0.001
8,Complication_Count,0.00 (0.00-1.00),1.00 (0.00-2.00),<0.001
9,Sex = F,237 (92.2%),137 (85.1%),0.032



Supplementary Table S1: Hyperparameters


,Model,Hyperparameters,Preprocessing
0,Logistic Regression,max_iter=1000; class_weight=balanced; solver=l...,Median imputation for numeric variables; Unkno...
1,Random Forest,n_estimators=300; random_state=42; class_weigh...,Median imputation for numeric variables; Unkno...
2,XGBoost,n_estimators=300; learning_rate=0.03; max_dept...,Median imputation for numeric variables; Unkno...



Supplementary Table S2: Confusion matrices


,Feature_Set,Model,TN,FP,FN,TP,Accuracy,Sensitivity,F1
0,Clinical-only,Logistic Regression,176,81,56,105,0.672,0.652,0.605
1,Lab-only,Logistic Regression,210,47,50,111,0.768,0.689,0.696
2,Lab + Clinical,Logistic Regression,208,49,49,112,0.766,0.696,0.696
3,Full,Logistic Regression,205,52,43,118,0.773,0.733,0.713
4,Full + Engineered,Logistic Regression,204,53,42,119,0.773,0.739,0.715



Supplementary Table S3: Calibration


,Feature_Set,Model,Pooled_AUROC_for_curve,Pooled_AUPRC_for_curve,Brier_score
0,Clinical-only,Logistic Regression,0.700,0.629,0.213
1,Lab-only,Logistic Regression,0.812,0.735,0.175
2,Lab + Clinical,Logistic Regression,0.814,0.739,0.173
3,Full,Logistic Regression,0.824,0.744,0.169
4,Full + Engineered,Logistic Regression,0.826,0.745,0.169



Saved corrected files:
Table1_baseline_characteristics_corrected.csv
Table2_model_performance_corrected.csv
Table2_model_performance_corrected_rounded.csv
Supplementary_Table_S1_hyperparameters.csv
Supplementary_Table_S2_confusion_matrices.csv
Supplementary_Table_S3_calibration.csv


In [4]:
Table2_display

,Feature_Set,Model,N_Raw_Features,CV_Design,AUROC_mean,AUROC_sd,AUPRC_mean,AUPRC_sd,Accuracy_mean,Accuracy_sd,Sensitivity_mean,Sensitivity_sd,F1_mean,F1_sd
9,Full,Logistic Regression,17,5-fold stratified CV repeated 10 times (50 spl...,0.840,0.032,0.769,0.047,0.765,0.039,0.709,0.084,0.698,0.054
10,Full,Random Forest,17,5-fold stratified CV repeated 10 times (50 spl...,0.837,0.037,0.783,0.053,0.778,0.031,0.630,0.068,0.685,0.048
12,Full + Engineered,Logistic Regression,23,5-fold stratified CV repeated 10 times (50 spl...,0.836,0.035,0.765,0.050,0.775,0.040,0.723,0.092,0.710,0.062
11,Full,XGBoost,17,5-fold stratified CV repeated 10 times (50 spl...,0.834,0.035,0.784,0.048,0.772,0.035,0.642,0.080,0.683,0.055
13,Full + Engineered,Random Forest,23,5-fold stratified CV repeated 10 times (50 spl...,0.833,0.036,0.776,0.051,0.784,0.031,0.648,0.064,0.697,0.047
14,Full + Engineered,XGBoost,23,5-fold stratified CV repeated 10 times (50 spl...,0.830,0.035,0.778,0.048,0.771,0.034,0.644,0.077,0.682,0.054
6,Lab + Clinical,Logistic Regression,13,5-fold stratified CV repeated 10 times (50 spl...,0.818,0.037,0.752,0.056,0.771,0.039,0.691,0.076,0.698,0.054
7,Lab + Clinical,Random Forest,13,5-fold stratified CV repeated 10 times (50 spl...,0.815,0.038,0.760,0.051,0.761,0.034,0.616,0.070,0.664,0.051
3,Lab-only,Logistic Regression,9,5-fold stratified CV repeated 10 times (50 spl...,0.812,0.040,0.745,0.059,0.765,0.044,0.682,0.081,0.690,0.062
4,Lab-only,Random Forest,9,5-fold stratified CV repeated 10 times (50 spl...,0.805,0.042,0.751,0.053,0.750,0.035,0.612,0.081,0.652,0.056


In [5]:
from google.colab import files

files.download("Table1_baseline_characteristics_corrected.csv")
files.download("Table2_model_performance_corrected_rounded.csv")
files.download("Supplementary_Table_S1_hyperparameters.csv")
files.download("Supplementary_Table_S2_confusion_matrices.csv")
files.download("Supplementary_Table_S3_calibration.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>